# Poisson Galaxy Shot-Noise Demo

Demonstrates the Poisson forward model used in `Gaussian_NPE_Poisson`:
what the noise looks like on a real Quijote simulation field,
and how it compares across different galaxy surveys.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch

from gaussian_npe import utils

# ── Cosmology / box constants (must match train.py) ─────────────────────────
BOX_PARAMS = {'box_size': 1000., 'grid_res': 128, 'h': 0.6711}
COSMO_PARAMS = {
    'h': 0.6711, 'Omega_b': 0.049, 'Omega_cdm': 0.2685,
    'n_s': 0.9624, 'non linear': 'halofit', 'sigma8': 0.834,
}

# ── Configure here ───────────────────────────────────────────────────────────
SAMPLE_PATH = '/path/to/Quijote_sample0.pt'   # <-- edit before running
N_BAR       = 5e-4    # galaxy number density [h³/Mpc³]  (DESI LRG-like)
GALAXY_BIAS = 1.5     # linear galaxy bias
SEED        = 42
# ─────────────────────────────────────────────────────────────────────────────

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
box     = utils.Power_Spectrum_Sampler(BOX_PARAMS, device=device)
V_voxel = (box.box_size / box.N) ** 3   # (Mpc/h)³ per voxel
N       = box.N

print(f'Box: {box.box_size} Mpc/h,  {N}³ voxels')
print(f'V_voxel = {V_voxel:.1f} (Mpc/h)³')
print(f'k_F = {box.k_F:.4f} h/Mpc,  k_Nyquist = {box.k_Nq:.4f} h/Mpc')
print(f'Device: {device}')

## Poisson forward model

Given the dark-matter overdensity field $\delta(\mathbf{r})$ at $z=0$,
a galaxy survey samples it as

$$N_\mathrm{gal}(\mathbf{r}) \sim \mathrm{Poisson}\!\left[\,\bar{N}\,\bigl(1 + b\,\delta(\mathbf{r})\bigr)^+\right]$$

where $\bar{N} = \bar{n}\,V_\mathrm{vox}$ is the mean galaxy count per voxel,
$b$ is the linear galaxy bias, and $(\cdot)^+$ denotes a ReLU clamp to prevent
negative means.

The rescaled observed field fed to the network is

$$x_\mathrm{eff}(\mathbf{r})
  = \frac{N_\mathrm{gal}(\mathbf{r})}{\bar{N}\,b} - \frac{1}{b}
  \approx \delta(\mathbf{r}) + \eta(\mathbf{r})$$

The shot-noise $\eta$ is approximately white (flat power spectrum) with

$$\sigma_\mathrm{eff} = \frac{1}{b\,\sqrt{\bar{N}}}, \qquad
  P_\mathrm{shot} = \sigma_\mathrm{eff}^2\,V_\mathrm{vox} = \frac{1}{b^2\,\bar{n}}$$

### Survey comparison

| Survey | $\bar{n}$ [h³/Mpc³] | $b$ | $\bar{N}$ | $\sigma_\mathrm{eff}$ | $P_\mathrm{shot}$ [(Mpc/h)³] |
|---|---|---|---|---|---|
| SDSS BOSS LRG  | 3×10⁻⁴ | 2.0 | 143 | 1.31 | 833  |
| DESI LRG       | 5×10⁻⁴ | 2.0 | 238 | 1.03 | 500  |
| DESI ELG       | 6×10⁻⁴ | 1.3 | 286 | 1.13 | 990  |
| Euclid H$\alpha$ | 2×10⁻³ | 1.5 | 952 | 0.69 | 222  |

In [ ]:
# ── Load sample ──────────────────────────────────────────────────────────────
sample   = torch.load(SAMPLE_PATH, weights_only=False)
delta_z0 = sample['delta_z0'].astype('f')       # (128,128,128) float32

print(f'delta_z0  shape : {delta_z0.shape}')
print(f'           min  : {delta_z0.min():.3f}')
print(f'           max  : {delta_z0.max():.3f}')
print(f'           std  : {delta_z0.std():.3f}')
print(f'           mean : {delta_z0.mean():.3e}  (should be ~0)')

In [ ]:
# ── Poisson noise helper ─────────────────────────────────────────────────────
def apply_poisson_noise(delta, n_bar, galaxy_bias, V_voxel, seed=None):
    """Apply the Poisson galaxy shot-noise forward model.

    Parameters
    ----------
    delta       : (N,N,N) numpy array, dark-matter overdensity at z=0
    n_bar       : float, galaxy number density [h³/Mpc³]
    galaxy_bias : float, linear galaxy bias
    V_voxel     : float, voxel volume (Mpc/h)³
    seed        : int or None

    Returns
    -------
    x_eff     : (N,N,N) float32, rescaled observed field
    sigma_eff : float, expected RMS of shot noise
    """
    rng    = np.random.default_rng(seed)
    N_bar  = n_bar * V_voxel
    N_mean = (N_bar * (1.0 + galaxy_bias * delta)).clip(min=0)
    N_obs  = rng.poisson(N_mean).astype('f')
    x_eff  = N_obs / (N_bar * galaxy_bias) - 1.0 / galaxy_bias
    sigma_eff = 1.0 / (galaxy_bias * N_bar ** 0.5)
    return x_eff, sigma_eff


# Apply for the configured parameters
x_eff, sigma_eff = apply_poisson_noise(delta_z0, N_BAR, GALAXY_BIAS, V_voxel, seed=SEED)
noise = x_eff - delta_z0

N_bar_val = N_BAR * V_voxel
print(f'n_bar        = {N_BAR:.2e} h³/Mpc³')
print(f'galaxy_bias  = {GALAXY_BIAS}')
print(f'N_bar (mean count/voxel) = {N_bar_val:.1f}')
print(f'sigma_eff (expected)     = {sigma_eff:.4f}')
print(f'std(noise)  (measured)   = {noise.std():.4f}')
print(f'Ratio measured/expected  = {noise.std()/sigma_eff:.4f}  (should be ~1)')

In [ ]:
# ── Field slice visualisation ─────────────────────────────────────────────────
# 3 rows × 3 cols: (clean δ_z0) / (noisy x_eff) / (noise η) × (xy / xz / yz)

fields     = [delta_z0, x_eff, noise]
row_labels = [r'$\delta_{z0}$ (clean)', r'$x_\mathrm{eff}$ (noisy)', r'$\eta = x_\mathrm{eff} - \delta_{z0}$ (noise)']
col_labels = ['xy  (z = N/2)', 'xz  (y = N/2)', 'yz  (x = N/2)']

# colour range: rows 0-1 share clean field range; row 2 uses ±3σ_eff
vmax_sig = np.percentile(np.abs(delta_z0), 99)
vmins = [-vmax_sig, -vmax_sig, -3 * sigma_eff]
vmaxs = [ vmax_sig,  vmax_sig,  3 * sigma_eff]

fig, axes = plt.subplots(3, 3, figsize=(12, 11))
fig.suptitle(
    f'Poisson noise  ($\\bar{{n}}={N_BAR:.0e}$, $b={GALAXY_BIAS}$, '
    f'$\\sigma_{{\\mathrm{{eff}}}}={sigma_eff:.2f}$)',
    fontsize=14,
)

for r, (field, vmin, vmax) in enumerate(zip(fields, vmins, vmaxs)):
    slices = [
        field[N // 2, :, :],   # xy
        field[:, N // 2, :],   # xz
        field[:, :, N // 2],   # yz
    ]
    for c, (sl, col_lbl) in enumerate(zip(slices, col_labels)):
        ax  = axes[r, c]
        im  = ax.imshow(sl, origin='lower', cmap='RdBu_r',
                        vmin=vmin, vmax=vmax,
                        extent=[0, box.box_size, 0, box.box_size])
        if r == 0:
            ax.set_title(col_lbl, fontsize=11)
        if c == 0:
            ax.set_ylabel(row_labels[r], fontsize=10)
        ax.set_xlabel('Mpc/h')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# ── Power spectrum comparison ────────────────────────────────────────────────
# Pylians power spectra of the three fields
k_sig,   pk_sig   = box.get_pk_pylians(delta_z0, MAS='PCS')
k_obs,   pk_obs   = box.get_pk_pylians(x_eff,    MAS='PCS')
k_noise, pk_noise = box.get_pk_pylians(noise,     MAS='PCS')

# CLASS linear P(k) at z=0 for reference
k_class = np.logspace(np.log10(box.k_F * 0.5), np.log10(box.k_Nq * 1.2), 200)
pk_lin  = utils.get_pk_class(COSMO_PARAMS, z=0, k=k_class)

# Theoretical shot noise level
P_shot = sigma_eff ** 2 * V_voxel

fig, ax = plt.subplots(figsize=(8, 5))

ax.loglog(k_sig,   pk_sig,   color='steelblue', lw=2,   label=r'$\delta_{z0}$ (clean)')
ax.loglog(k_obs,   pk_obs,   color='darkorange', lw=2,  label=r'$x_\mathrm{eff}$ (noisy)')
ax.loglog(k_noise, pk_noise, color='seagreen',   lw=2,  label=r'noise $\eta$')
ax.loglog(k_class, pk_lin,   color='0.55', lw=1.5, ls='--', label='CLASS linear (z=0)')
ax.axhline(P_shot, color='firebrick', lw=1.5, ls=':', label=f'$P_{{\\rm shot}}={P_shot:.0f}$ (Mpc/h)³')
ax.axvline(box.k_Nq, color='0.4', lw=1, ls=':')
ax.text(box.k_Nq * 1.05, ax.get_ylim()[0] * 2, '$k_\\mathrm{Nq}$', color='0.4', fontsize=9)

ax.set_xlabel('$k$  [h/Mpc]')
ax.set_ylabel('$P(k)$  [(Mpc/h)³]')
ax.set_title(
    f'Power spectra  ($\\bar{{n}}={N_BAR:.0e}$, $b={GALAXY_BIAS}$, '
    f'$\\sigma_{{\\rm eff}}={sigma_eff:.2f}$)'
)
ax.legend(fontsize=9)
ax.set_xlim(k_class[0], k_class[-1])
plt.tight_layout()
plt.show()

print(f'P_shot (theoretical) = {P_shot:.1f}  (Mpc/h)³')
print(f'P_shot = 1 / (b² · n_bar) = {1/(GALAXY_BIAS**2 * N_BAR):.1f}  (Mpc/h)³')

In [ ]:
# ── Survey comparison ─────────────────────────────────────────────────────────
SURVEYS = [
    ('SDSS BOSS LRG',  3e-4, 2.0),
    ('DESI LRG',       5e-4, 2.0),
    ('DESI ELG',       6e-4, 1.3),
    ('Euclid Hα',      2e-3, 1.5),
]

print(f'{"Survey":<18}  {"n_bar [h³/Mpc³]":>16}  {"b":>4}  {"N_bar":>6}  '
      f'{"σ_eff":>6}  {"P_shot [(Mpc/h)³]":>18}')
print('-' * 78)
for name, n_bar, b in SURVEYS:
    N_bar_v   = n_bar * V_voxel
    sig       = 1.0 / (b * N_bar_v ** 0.5)
    P_s       = sig ** 2 * V_voxel
    print(f'{name:<18}  {n_bar:>16.2e}  {b:>4.1f}  {N_bar_v:>6.0f}  {sig:>6.2f}  {P_s:>18.0f}')

# ── Noise P(k) sweep ─────────────────────────────────────────────────────────
colours = ['steelblue', 'darkorange', 'seagreen', 'mediumpurple']

fig, ax = plt.subplots(figsize=(8, 5))

# Clean signal P(k) as background reference
ax.loglog(k_sig, pk_sig, color='0.7', lw=1.5, ls='--', label=r'$\delta_{z0}$ (signal)')

for (name, n_bar, b), col in zip(SURVEYS, colours):
    x_s, sig_s = apply_poisson_noise(delta_z0, n_bar, b, V_voxel, seed=SEED)
    noise_s    = x_s - delta_z0
    k_s, pk_s  = box.get_pk_pylians(noise_s, MAS='PCS')
    P_s        = sig_s ** 2 * V_voxel
    ax.loglog(k_s, pk_s, color=col, lw=1.8, label=f'{name}  (σ={sig_s:.2f})')
    ax.axhline(P_s, color=col, lw=0.8, ls=':')

ax.axvline(box.k_Nq, color='0.4', lw=1, ls=':')
ax.set_xlabel('$k$  [h/Mpc]')
ax.set_ylabel('$P(k)$  [(Mpc/h)³]')
ax.set_title('Shot-noise $P(k)$ for different galaxy surveys\n(dotted lines = theoretical $P_\\mathrm{shot}$)')
ax.legend(fontsize=9)
ax.set_xlim(k_s[0], k_s[-1])
plt.tight_layout()
plt.show()